# Étape 2 · Nettoyage

Lecture du fichier du challenge, contrôles de cohérence, sources externes.

À lancer dans le projet complet (archive `05-interpretation_D1_scientist_projet-complet_v1.zip`), avec les données brutes placées comme le dit `data/raw/LISEZ-MOI.md`.

In [1]:
import csv
import glob
import os
import re
import sys
import time
import unicodedata
from pathlib import Path

from IPython.display import Image, display
import openpyxl
import pandas as pd

root = Path.cwd()
while not (root / "scripts").is_dir() and root != root.parent:
    root = root.parent
os.chdir(root)
sys.path.insert(0, "scripts")

In [2]:
def show_figures(since, folders=("analyse/figures", "analyse", "analyse/fiches"), limit=8):
    views = sorted((p for d in folders for p in Path(d).glob("*.png") if p.stat().st_mtime >= since), key=lambda p: p.name)
    for p in views[:limit]:
        print(p)
        display(Image(filename=str(p), width=760))

## 1. Lecture du fichier du challenge

`scripts/parse_presco.py`

In [3]:
YEARS = [f"{y}-{y + 1}" for y in range(2015, 2025)]
TABLES = [
    ("P1", "prescolaire", "effectifs", "eleves", 1, 17, "wide", ["region"]),
    ("P2", "prescolaire", "effectifs", "eleves", 19, 25, "wide", ["statut"]),
    ("P3", "prescolaire", "effectifs", "eleves", 27, 33, "wide", ["milieu"]),
    ("P4", "prescolaire", "effectifs", "eleves", 36, 52, "wide", ["statut", "niveau"]),
    ("P5", "prescolaire", "TBS", "%", 55, 72, "wide", ["region"]),
    ("P6", "prescolaire", "IPS_TBS", "ratio F/G", 75, 90, "plain", ["region"]),
    ("P7", "prescolaire", "structures", "structures", 93, 147, "plain", ["region", "statut"]),
    ("P8", "prescolaire", "salles", "salles", 150, 204, "plain", ["region", "statut"]),
    ("E1", "primaire", "effectifs", "eleves", 206, 250, "long", ["region"]),
    ("E2", "primaire", "effectifs", "eleves", 276, 286, "long", ["statut"]),
    ("E3", "primaire", "effectifs", "eleves", 349, 359, "long", ["milieu"]),
    ("E4", "primaire", "effectifs", "eleves", 447, 493, "long", ["statut", "niveau"]),
    ("E5", "primaire", "nouveaux_inscrits_CP1", "eleves", 562, 605, "long", ["region"]),
    ("E6", "primaire", "TBA", "%", 641, 657, "wide", ["region"]),
    ("E7", "primaire", "TBS", "%", 701, 717, "wide", ["region"]),
    ("E8", "primaire", "TAP", "%", 809, 825, "wide", ["region"]),
    ("E9", "primaire", "reussite_CEP", "%", 894, 937, "long", ["region"]),
    ("E10", "primaire", "IPS_TBA", "ratio F/G", 1061, 1076, "plain", ["region"]),
    ("E11", "primaire", "IPS_TBS", "ratio F/G", 1189, 1204, "plain", ["region"]),
    ("E12", "primaire", "IPS_TAP", "ratio F/G", 1314, 1329, "plain", ["region"]),
    ("E13", "primaire", "ecoles", "ecoles", 1417, 1460, "plain", ["region", "statut"]),
]
SEX = {"G": "G", "F": "F", "T": "T", "GARCONS": "G", "FILLES": "F", "TOTAL": "T",
       "GARÇONS": "G"}
NATIONAL = {"TOTAL GÉNÉRAL", "TOTAL GENERAL", "NATIONAL", "BURKINA FASO", "TOTAL"}
HEADERS = {"REGION", "RÉGION", "RÉGIONS", "REGIONS", "STATUT", "MILIEU"}

In [4]:
def number(cell):
    c = cell.replace(" ", "").replace(" ", "").replace(",", ".").strip()
    if c == "":
        return None
    if not re.fullmatch(r"-?\d+(\.\d+)?", c):
        raise ValueError(f"not a number: {cell!r}")
    return float(c)

In [5]:
def is_number(cell):
    try:
        return number(cell) is not None
    except ValueError:
        return False

In [6]:
def norm_region(name):
    n = name.strip().upper()
    if n in NATIONAL:
        return "NATIONAL"
    return "HAUTS-BASSINS" if n == "HAUT-BASSINS" else n

In [7]:
def norm_label(kind, value):
    v = value.strip()
    if kind == "region":
        return norm_region(v)
    if kind == "statut":
        return {"PUBLIC": "Public", "PRIVÉ": "Prive", "PRIVE": "Prive", "COMMUNAUTAIRE": "Communautaire",
                "TOTAL": "Total"}.get(v.upper(), v)
    if kind == "milieu":
        return {"URBAIN": "Urbain", "RURAL": "Rural", "INDETERMINÉ": "Indetermine",
                "INDETERMINE": "Indetermine", "TOTAL GÉNÉRAL": "Total", "NATIONAL": "Total"}.get(v.upper(), v)
    return v

In [8]:
def labels_from_row(cells, kinds, carried):
    out = {}
    for i, kind in enumerate(kinds):
        raw = cells[i].strip() if i < len(cells) else ""
        out[kind] = raw if raw else carried.get(kind, "")
    first = kinds[0]
    raw0 = cells[0].strip() if cells else ""
    if raw0.upper().startswith("TOTAL ") and raw0.upper() not in NATIONAL:
        rest = raw0[6:]
        if first == "region":
            out["region"], out[kinds[1]] = rest, "Total"
        else:
            out[first], out[kinds[1]] = rest, "Total"
    elif raw0.upper() in NATIONAL:
        out[first] = "NATIONAL" if first == "region" else "Total"
        for k in kinds[1:]:
            if not (cells[kinds.index(k)].strip() if kinds.index(k) < len(cells) else ""):
                out[k] = "Total"
    return out

In [9]:
def xlsx_lines(path):
    """The platform's .xlsx, read as the tab-separated export it comes from: one line per row, same layout."""
    book = openpyxl.load_workbook(path, data_only=True)
    sheet = book.worksheets[0]
    lines = []
    for row in sheet.iter_rows():
        cells = []
        for c in row:
            v = c.value
            if v is None:
                cells.append("")
            elif isinstance(v, float) and "%" in (c.number_format or ""):
                cells.append(f"{v * 100:.1f}")
            elif isinstance(v, float) and v.is_integer():
                cells.append(str(int(v)))
            else:
                cells.append(str(v))
        lines.append("\t".join(cells))
    return lines

In [10]:
def find_source(path):
    """Presco.txt, or else the challenge .xlsx placed in the same folder."""
    if os.path.exists(path):
        return path
    found = sorted(glob.glob(os.path.join(os.path.dirname(path) or ".", "*.xlsx")))
    if not found:
        raise SystemExit(f"Fichier du challenge introuvable : placer {path} ou le fichier .xlsx de la plateforme dans {os.path.dirname(path)}/")
    return found[0]

In [11]:
def parse(path):
    if path.lower().endswith(".xlsx"):
        lines = xlsx_lines(path)
    else:
        lines = open(path, encoding="utf-8", newline="").read().split("\r\n")
    rows = []
    for tid, cycle, ind, unit, a, b, layout, kinds in TABLES:
        block = [lines[i].split("\t") for i in range(a - 1, b)]
        carried = {}
        if layout == "wide":
            gft = next(i for i, r in enumerate(block) if [c.strip() for c in r[:6]].count("G") >= 1
                       and any(c.strip() == "F" for c in r))
            start = next(j for j, c in enumerate(block[gft]) if c.strip() == "G")
            ntrip = sum(1 for c in block[gft][start:] if c.strip() == "G")
            years = YEARS[-ntrip:]
            for r in block[gft + 1:]:
                if not any(c.strip() for c in r):
                    continue
                lab = labels_from_row(r, kinds, carried)
                carried = {k: v for k, v in lab.items()}
                for t, year in enumerate(years):
                    for s, sex in enumerate("GFT"):
                        j = start + 3 * t + s
                        v = number(r[j]) if j < len(r) else None
                        if v is not None:
                            rows.append((tid, cycle, ind, unit, lab, sex, year, v))
        else:
            nlab = len(kinds) + (1 if layout == "long" else 0)
            for r in block:
                cells = [c.strip() for c in r]
                vals = cells[nlab:nlab + 10]
                if not cells or len(vals) < 10 or not all(is_number(v) for v in vals):
                    continue
                if cells[0].upper() in HEADERS or (layout == "long" and cells[len(kinds)].upper() == "SEXE"):
                    continue
                lab = labels_from_row(cells, kinds, carried)
                carried = {k: v for k, v in lab.items()}
                if layout == "long":
                    sraw = cells[len(kinds)].upper() or carried.get("_sexe", "")
                    sex = SEX.get(sraw)
                    if sex is None:
                        raise ValueError(f"{tid}: unknown sex label {cells[len(kinds)]!r}")
                else:
                    sex = "T"
                for year, v in zip(YEARS, vals):
                    rows.append((tid, cycle, ind, unit, lab, sex, year, number(v)))
    return rows

In [12]:
def main():
    src, dst = find_source(sys.argv[1]), sys.argv[2]
    print("fichier du challenge lu :", src)
    rows = parse(src)
    known = {(r[0], r[4].get("region", ""), r[5], r[6]): r[7] for r in rows}
    checks = [(("E1", "TOTAL GENERAL", "T", "2024-2025"), 3308096), (("E8", "TOTAL GENERAL", "T", "2024-2025"), 51.6)]
    for key_, expected in checks:
        got = next((v for k, v in known.items() if k[0] == key_[0] and norm_label("region", k[1]) == "NATIONAL" and k[2:] == key_[2:]), None)
        if got is None or abs(got - expected) > 0.05:
            raise SystemExit(f"Lecture incorrecte du fichier du challenge ({key_[0]} {key_[3]} : {got} au lieu de {expected}). "
                             "Enregistrer le fichier depuis Excel en « Texte (séparateur : tabulation) » sous data/raw/Presco.txt.")
    with open(dst, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["table", "cycle", "indicateur", "unite", "region", "statut", "milieu", "niveau",
                    "sexe", "annee", "valeur"])
        for tid, cycle, ind, unit, lab, sex, year, v in rows:
            region = norm_label("region", lab["region"]) if "region" in lab else "NATIONAL"
            status = norm_label("statut", lab.get("statut", "")) or "Total"
            area = norm_label("milieu", lab.get("milieu", "")) or "Total"
            tier = lab.get("niveau", "") or "Total"
            if tier.upper() == "TOTAL":
                tier = "Total"
            w.writerow([tid, cycle, ind, unit, region, status, area, tier, sex, year,
                        int(v) if v.is_integer() else v])
    print(f"{len(rows)} valeurs écrites dans {dst}")

In [13]:
run_parse_presco = main
t0 = time.time()
sys.argv = ['parse_presco.py', 'data/raw/Presco.txt', 'data/tidy/presco_tidy.csv']
run_parse_presco()
show_figures(t0)

fichier du challenge lu : data/raw/Presco.txt
6621 valeurs écrites dans data/tidy/presco_tidy.csv


## 2. Contrôles de cohérence

`scripts/check_presco.py`

In [14]:
COUNT_TABLES = ["P1", "P2", "P3", "P4", "P7", "P8", "E1", "E2", "E3", "E4", "E5", "E13"]
RATE_IPS = [("E6", "E10", "TBA"), ("E7", "E11", "TBS"), ("E8", "E12", "TAP"), ("P5", "P6", "TBS préscolaire")]

In [15]:
def section(title):
    print(f"\n=== {title} ===")

In [16]:
def report(name, tested, failures, show=8):
    status = "OK" if tested and failures.empty else ("RIEN TESTÉ" if not tested else f"{len(failures)} ÉCART(S)")
    print(f"- {name} : {tested} cas testés -> {status}")
    if tested and not failures.empty:
        print(failures.head(show).to_string(index=False))

In [17]:
def main():
    d = pd.read_csv(sys.argv[1])
    keys = ["table", "region", "statut", "milieu", "niveau", "annee"]

    section("1. Garçons + filles = total (tableaux d'effectifs)")
    c = d[d.table.isin(COUNT_TABLES)].pivot_table(index=keys, columns="sexe", values="valeur").dropna()
    if {"G", "F", "T"} <= set(c.columns):
        c = c.dropna(subset=["G", "F", "T"])
        bad = c.loc[(c["G"] + c["F"] - c["T"]).abs() > 0.5].reset_index()
        bad["ecart"] = bad["G"] + bad["F"] - bad["T"]
        report("G + F = T", len(c), bad[keys + ["G", "F", "T", "ecart"]])

    section("2. Somme des régions = national")
    for t in ["P1", "P7", "P8", "E1", "E5", "E13"]:
        x = d[(d.table == t)]
        reg = x[x.region != "NATIONAL"].groupby(["statut", "sexe", "annee"]).valeur.sum()
        nat = x[x.region == "NATIONAL"].set_index(["statut", "sexe", "annee"]).valeur
        j = pd.concat([reg.rename("somme_regions"), nat.rename("national")], axis=1).dropna()
        j["ecart"] = j.somme_regions - j.national
        report(f"{t} ({x.indicateur.iloc[0]})", len(j), j[j.ecart.abs() > 0.5].reset_index())

    section("3. Le même total national dans les différents tableaux")
    def nat(t, **f):
        x = d[(d.table == t) & (d.sexe == "T")]
        for k, v in f.items():
            x = x[x[k] == v]
        return x.groupby("annee").valeur.sum()
    pairs = {
        "primaire : régions (E1) / statut (E2)": (nat("E1", region="NATIONAL"), nat("E2", statut="Total")),
        "primaire : régions (E1) / milieu (E3)": (nat("E1", region="NATIONAL"), nat("E3", milieu="Total")),
        "primaire : régions (E1) / niveaux (E4)": (nat("E1", region="NATIONAL"), nat("E4", statut="Total", niveau="Total")),
        "préscolaire : régions (P1) / statut (P2)": (nat("P1", region="NATIONAL"), nat("P2", statut="Total")),
        "préscolaire : régions (P1) / milieu (P3)": (nat("P1", region="NATIONAL"), nat("P3", milieu="Total")),
        "préscolaire : régions (P1) / niveaux (P4)": (nat("P1", region="NATIONAL"), nat("P4", statut="Total", niveau="Total")),
    }
    for name, (a, b) in pairs.items():
        j = pd.concat([a.rename("A"), b.rename("B")], axis=1).dropna()
        j["ecart"] = j.A - j.B
        report(name, len(j), j[j.ecart.abs() > 0.5].reset_index())

    section("4. Somme des statuts et des niveaux = total")
    for t, parts in [("E2", ["Public", "Prive"]), ("E13", ["Public", "Prive"]),
                     ("P2", ["Public", "Prive", "Communautaire"]), ("P7", ["Public", "Prive", "Communautaire"]),
                     ("P8", ["Public", "Prive", "Communautaire"])]:
        x = d[d.table == t]
        s = x[x.statut.isin(parts)].groupby(["region", "sexe", "annee"]).valeur.sum()
        tot = x[x.statut == "Total"].set_index(["region", "sexe", "annee"]).valeur
        j = pd.concat([s.rename("somme"), tot.rename("total")], axis=1).dropna()
        j["ecart"] = j.somme - j.total
        report(f"{t} : statuts", len(j), j[j.ecart.abs() > 0.5].reset_index())
    x = d[(d.table == "E4") & (d.statut.isin(["Public", "Prive"]))]
    s = x[x.niveau != "Total"].groupby(["statut", "sexe", "annee"]).valeur.sum()
    tot = x[x.niveau == "Total"].set_index(["statut", "sexe", "annee"]).valeur
    j = pd.concat([s.rename("somme"), tot.rename("total")], axis=1).dropna()
    j["ecart"] = j.somme - j.total
    report("E4 : CP1 + ... + CM2 = total", len(j), j[j.ecart.abs() > 0.5].reset_index())

    section("5. Indice de parité = taux filles / taux garçons")
    for rt, it, name in RATE_IPS:
        r = d[d.table == rt].pivot_table(index=["region", "annee"], columns="sexe", values="valeur")
        ips = d[d.table == it].set_index(["region", "annee"]).valeur.rename("IPS_publie")
        j = pd.concat([(r.F / r.G).rename("F_sur_G"), ips], axis=1).dropna()
        j["ecart"] = (j.F_sur_G - j.IPS_publie).round(3)
        report(f"{name} ({rt} / {it}), tolérance 0,02", len(j), j[j.ecart.abs() > 0.02].reset_index())

    section("6. Lignes recopiées d'une région à l'autre (mêmes valeurs G, F, T la même année)")
    tested, found = 0, []
    for (t, yr), x in d[d.region != "NATIONAL"].groupby(["table", "annee"]):
        g = x.groupby(["region", "statut", "niveau", "milieu"]).valeur.apply(lambda v: tuple(sorted(v.round(6))))
        tested += len(g)
        dup = g[g.duplicated(keep=False) & g.map(lambda v: any(abs(z) > 0 for z in v) and len(v) > 0)]
        for vec, grp in dup.groupby(dup):
            regs = sorted(set(i[0] for i in grp.index))
            if len(regs) > 1 and len(vec) >= 2:
                found.append({"table": t, "annee": yr, "regions": " = ".join(regs), "valeurs": vec[:3]})
    report("valeurs identiques entre régions", tested, pd.DataFrame(found))

    section("7. Sauts de plus de 40 % d'une année à l'autre (effectifs et écoles par région)")
    x = d[d.table.isin(["P1", "E1", "E5", "E13"]) & (d.sexe == "T") & (d.statut == "Total")].sort_values("annee")
    x = x.assign(prev=x.groupby(["table", "region"]).valeur.shift())
    x["variation_%"] = ((x.valeur / x.prev - 1) * 100).round(1)
    jumps = x[x["variation_%"].abs() > 40][["table", "indicateur", "region", "annee", "prev", "valeur", "variation_%"]]
    report("variations > 40 %", int(x.prev.notna().sum()), jumps, show=40)

    section("8. Taux hors de l'ordre de grandeur attendu")
    lim = {"TAP": 100, "TBS": 130, "TBA": 130, "reussite_CEP": 100}
    r = d[d.indicateur.isin(lim) & (d.cycle == "primaire")]
    over = r[r.apply(lambda z: z.valeur > lim[z.indicateur], axis=1)]
    report("TAP > 100, TBS ou TBA > 130, CEP > 100", len(r), over[["table", "indicateur", "region", "sexe", "annee", "valeur"]], show=20)

In [18]:
run_check_presco = main
t0 = time.time()
sys.argv = ['check_presco.py', 'data/tidy/presco_tidy.csv']
run_check_presco()
show_figures(t0)


=== 1. Garçons + filles = total (tableaux d'effectifs) ===
- G + F = T : 827 cas testés -> OK

=== 2. Somme des régions = national ===
- P1 (effectifs) : 30 cas testés -> OK
- P7 (structures) : 10 cas testés -> OK
- P8 (salles) : 10 cas testés -> OK
- E1 (effectifs) : 30 cas testés -> OK
- E5 (nouveaux_inscrits_CP1) : 30 cas testés -> OK
- E13 (ecoles) : 30 cas testés -> OK

=== 3. Le même total national dans les différents tableaux ===
- primaire : régions (E1) / statut (E2) : 10 cas testés -> OK
- primaire : régions (E1) / milieu (E3) : 10 cas testés -> OK
- primaire : régions (E1) / niveaux (E4) : 10 cas testés -> OK
- préscolaire : régions (P1) / statut (P2) : 10 cas testés -> OK
- préscolaire : régions (P1) / milieu (P3) : 8 cas testés -> OK
- préscolaire : régions (P1) / niveaux (P4) : 10 cas testés -> OK

=== 4. Somme des statuts et des niveaux = total ===
- E2 : statuts : 30 cas testés -> OK
- E13 : statuts : 140 cas testés -> OK
- P2 : statuts : 30 cas testés -> OK
- P7 : sta

- valeurs identiques entre régions : 2990 cas testés -> 1 ÉCART(S)
table     annee           regions                        valeurs
   P5 2019-2020 SAHEL = SUD-OUEST (1.674571, 1.710186, 1.743707)

=== 7. Sauts de plus de 40 % d'une année à l'autre (effectifs et écoles par région) ===
- variations > 40 % : 504 cas testés -> 30 ÉCART(S)
table            indicateur            region     annee     prev   valeur  variation_%
   P1             effectifs   PLATEAU CENTRAL 2016-2017   1887.0   2718.0         44.0
   P1             effectifs            CENTRE 2016-2017  18857.0  28615.0         51.7
   P1             effectifs        CENTRE-SUD 2016-2017   2435.0    925.0        -62.0
   P1             effectifs            CENTRE 2017-2018  28615.0  41738.0         45.9
   P1             effectifs        CENTRE-SUD 2017-2018    925.0   1428.0         54.4
   P1             effectifs        CENTRE-SUD 2018-2019   1428.0   3775.0        164.4
  E13                ecoles             SAHEL 2018-20

## 3. Sources externes : faim, EDS, déplacés

`scripts/prepare_facteurs.py`

In [19]:
RAW = "data/raw/externe/"
OUT = "data/externe_propre/"

In [20]:
def key(name):
    s = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    return "".join(ch for ch in s.upper() if ch.isalnum())

In [21]:
DHS_NAMES = {"Boucle de Mouhoun": "BOUCLEDUMOUHOUN", "Centre (including Ouagadougou)": "CENTRE",
             "Hauts Bassins": "HAUTSBASSINS"}

In [22]:
def food():
    df = pd.read_excel(RAW + "hdx_cadre_harmonise/cadre_harmonise_caf_ipc_mars26.xlsx")
    b = df[(df.adm0_name == "Burkina Faso") & (df.usethisperiod == "Y")].copy()
    b["cle"] = b.adm1_name.map(key)
    start = b.reference_year - (b.reference_label != "Sep-Dec").astype(int)
    b["annee"] = start.astype(str) + "-" + (start + 1).astype(str)
    per = b.groupby(["cle", "annee", "reference_label"])[["phase35", "population"]].sum().reset_index()
    per["pct"] = per.phase35 / per.population * 100
    out = per.groupby(["cle", "annee"]).agg(faim_phase3plus_pct=("pct", "mean"), periodes=("pct", "size"),
                                             population_ch=("population", "max")).reset_index()
    out = out[out.annee >= "2014-2015"]
    assert out.cle.nunique() == 13, sorted(out.cle.unique())
    out.round(2).to_csv(OUT + "faim_cadre_harmonise_region_annee.csv", index=False)
    with open(OUT + "faim_cadre_harmonise_region_annee.SOURCE.txt", "w") as f:
        f.write("# Source : Cadre Harmonisé (CILSS / RPCA), consolidation HDX « cadre-harmonise », fichier "
                "cadre_harmonise_caf_ipc_mars26.xlsx, licence CC BY. Provinces sommées par ancienne région. "
                "Indicateur : part de la population en phase 3 ou plus, moyenne des périodes d'une année scolaire "
                "(sept.-déc. de t, janv.-mai et juin-août de t+1). Le Burkina n'apparaît plus après juin-août 2024 : "
                "dernière année scolaire couverte 2023-2024 (sans sept.-déc. 2024).\n")
    return out

In [23]:
def families():
    rows = []
    for f in ["select-education-indicators_subnational_bfa.csv", "mics-indicators_subnational_bfa.csv"]:
        d = pd.read_csv(RAW + "hdx_dhs/" + f)
        rows.append(d[d.SurveyYear.astype(str) == "2021"])
    d = pd.concat(rows)
    names = {"Women with no education": "meres_sans_instruction_pct",
             "Age specific fertility rate: 15-19": "fecondite_15_19_pour_mille",
             "Gender parity index for net primary school attendance": "ipp_frequentation_primaire"}
    d = d[d.Indicator.isin(names) & (d.CharacteristicLabel != "Ouagadougou")].drop_duplicates(
        ["Indicator", "CharacteristicLabel"])
    d["cle"] = d.CharacteristicLabel.map(lambda s: DHS_NAMES.get(s, key(s)))
    out = d.pivot_table(index="cle", columns="Indicator", values="Value").rename(columns=names).reset_index()
    assert len(out) == 13, out.cle.tolist()
    out.to_csv(OUT + "eds_2021_regions.csv", index=False)
    with open(OUT + "eds_2021_regions.SOURCE.txt", "w") as f:
        f.write("# Source : Enquête démographique et de santé 2021 (INSD / The DHS Program), indicateurs régionaux, "
                "HDX « dhs-subnational-data-for-burkina-faso ». « Centre » = ligne « Centre (including Ouagadougou) ». "
                "« meres_sans_instruction_pct » = femmes de 15-49 ans sans instruction. Limite : la collecte n'a pas "
                "couvert toutes les zones d'insécurité (Sahel, Est notamment).\n")
    return out

In [24]:
def displaced(food_df):
    x = pd.read_excel(RAW + "hdx_pdi_conasur/situation_pdi_31122022.xlsx", "STATISTIQUES")
    x = x.rename(columns={x.columns[0]: "region", x.columns[1]: "pdi"}).dropna(subset=["pdi"])
    x = x[pd.to_numeric(x.pdi, errors="coerce").notna() & (x.region != "BURKINA FASO")]
    x["cle"] = x.region.map(key)
    pop = food_df[food_df.annee == "2022-2023"].set_index("cle").population_ch
    x["pdi_pct_population"] = x.pdi.astype(float) / x.cle.map(pop) * 100
    assert len(x) == 13 and x.pdi_pct_population.notna().all()
    x[["cle", "pdi", "pdi_pct_population"]].round(2).to_csv(OUT + "pdi_conasur_2022_regions.csv", index=False)
    with open(OUT + "pdi_conasur_2022_regions.SOURCE.txt", "w") as f:
        f.write("# Source : CONASUR, enregistrement des personnes déplacées internes, situation au 31 décembre 2022, "
                "feuille STATISTIQUES (HDX « situation-des-personnes-deplacees-internes », licence CC BY). Déplacés "
                "comptés dans la région d'ACCUEIL. Dénominateur : population du Cadre Harmonisé 2022. Dernière série "
                "régionale publiée ; le gouvernement a cessé la publication en 2023.\n")
    return x

In [25]:
def main():
    f = food()
    print("Faim (phase 3+, %) :")
    print(f.pivot(index="cle", columns="annee", values="faim_phase3plus_pct").round(1).to_string())
    print(families().round(1).to_string(index=False))
    print(displaced(f).round(1).to_string(index=False))

In [26]:
run_prepare_facteurs = main
t0 = time.time()
sys.argv = ['prepare_facteurs.py']
run_prepare_facteurs()
show_figures(t0)

Faim (phase 3+, %) :
annee            2014-2015  2015-2016  2016-2017  2017-2018  2018-2019  2019-2020  2020-2021  2021-2022  2022-2023  2023-2024
cle                                                                                                                          
BOUCLEDUMOUHOUN        1.5        0.7        0.8        0.3        1.2        5.6        5.5        6.7       15.2        8.4
CASCADES               0.0        0.0        0.3        0.0        1.3        0.7        2.8        2.1        3.7        1.6
CENTRE                 0.0        0.0        1.0        0.2        2.0        2.7        5.3        8.0        4.7        1.7
CENTREEST              0.1        0.2        0.2        0.1        0.3        4.3        5.9        3.9        7.1        8.7
CENTRENORD             1.9        2.6        1.1        6.2        3.3       31.9       28.8       24.2       21.4       21.7
CENTREOUEST            0.1        2.6        0.5        6.8        0.9        1.8        5.5     